# Assignment 4: MLOps & Model Deployment
## NYC Taxi Tip Prediction Service

## Overview
This notebook documents the full MLOps pipeline for deploying
the taxi tip prediction model from Assignment 2, including:
- Part 1: Experiment tracking with MLflow
- Part 2: Model serving with FastAPI  
- Part 3: Containerization with Docker

In [1]:
# ============================================================
# IMPORTS & CONFIGURATION
# ============================================================
import mlflow
import mlflow.sklearn
import mlflow.pytorch
import joblib
import json
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

mlflow.set_tracking_uri("http://localhost:5000")

print(" Libraries imported!")
print(f"   MLflow version: {mlflow.__version__}")
print(f"   Tracking URI: {mlflow.get_tracking_uri()}")

 Libraries imported!
   MLflow version: 3.11.1
   Tracking URI: http://localhost:5000


In [2]:
# ============================================================
#LOADING SAVED MODELS FROM ASSIGNMENT 2
# ============================================================

import os

# Check files exist
model_path   = 'models/taxi_tip_model.pkl'
scaler_path  = 'models/scaler.pkl'
features_path = 'models/feature_columns.json'

for path in [model_path, scaler_path, features_path]:
    exists = "✅" if os.path.exists(path) else " MISSING"
    print(f"   {exists} {path}")

rf_model       = joblib.load(model_path)
scaler         = joblib.load(scaler_path)
feature_columns = json.load(open(features_path))

print(f"\n Models loaded successfully!")
print(f"   Model type: {type(rf_model).__name__}")
print(f"   Number of features: {len(feature_columns)}")
print(f"   Features: {feature_columns[:5]}... (showing first 5)")

   ✅ models/taxi_tip_model.pkl
   ✅ models/scaler.pkl
   ✅ models/feature_columns.json

 Models loaded successfully!
   Model type: RandomForestRegressor
   Number of features: 29
   Features: ['passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax']... (showing first 5)


In [4]:
# ============================================================
#CREATING EVALUATION DATA
# ============================================================


np.random.seed(42)
n_samples = 1000


sample_data = pd.DataFrame({
    'passenger_count':        np.random.randint(1, 5, n_samples).astype(float),
    'trip_distance':          np.random.exponential(3, n_samples),
    'fare_amount':            np.random.uniform(5, 50, n_samples),
    'extra':                  np.random.choice([0, 0.5, 1], n_samples).astype(float),
    'mta_tax':                np.full(n_samples, 0.5),
    'tolls_amount':           np.random.choice([0, 0, 0, 6.55], n_samples).astype(float),
    'improvement_surcharge':  np.full(n_samples, 0.3),
    'congestion_surcharge':   np.random.choice([0, 2.5], n_samples).astype(float),
    'Airport_fee':            np.random.choice([0, 0, 1.25], n_samples).astype(float),
    'trip_duration_minutes':  np.random.uniform(3, 45, n_samples),
    'trip_speed_mph':         np.random.uniform(5, 30, n_samples),
    'pickup_hour':            np.random.randint(0, 24, n_samples).astype(float),
    'pickup_day_of_week':     np.random.randint(0, 7, n_samples).astype(float),
    'is_weekend':             np.random.randint(0, 2, n_samples).astype(float),
    'log_trip_distance':      np.random.uniform(0, 3, n_samples),
    'fare_per_mile':          np.random.uniform(3, 15, n_samples),
    'fare_per_minute':        np.random.uniform(0.5, 3, n_samples),
    'pickup_boro_Bronx':      np.zeros(n_samples),
    'pickup_boro_Brooklyn':   np.zeros(n_samples),
    'pickup_boro_Manhattan':  np.ones(n_samples),
    'pickup_boro_Other':      np.zeros(n_samples),
    'pickup_boro_Queens':     np.zeros(n_samples),
    'pickup_boro_Staten Island': np.zeros(n_samples),
    'dropoff_boro_Bronx':     np.zeros(n_samples),
    'dropoff_boro_Brooklyn':  np.zeros(n_samples),
    'dropoff_boro_Manhattan': np.ones(n_samples),
    'dropoff_boro_Other':     np.zeros(n_samples),
    'dropoff_boro_Queens':    np.zeros(n_samples),
    'dropoff_boro_Staten Island': np.zeros(n_samples),
})


sample_data = sample_data[feature_columns]


X_scaled = pd.DataFrame(
    scaler.transform(sample_data),
    columns=feature_columns
)


y_true = sample_data['fare_amount'] * np.random.uniform(0.1, 0.3, n_samples)

print(" Evaluation data created!")
print(f"   Samples: {len(X_scaled):,}")
print(f"   Features: {X_scaled.shape[1]}")
print(f"   Sample tip amounts: ${y_true.min():.2f} - ${y_true.max():.2f}")

 Evaluation data created!
   Samples: 1,000
   Features: 29
   Sample tip amounts: $0.60 - $14.87


In [5]:
# ============================================================
# LOG MODELS TO MLFLOW
# ============================================================

def log_regression_metrics(y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mlflow.log_metric("mae",  round(mae,  4))
    mlflow.log_metric("rmse", round(rmse, 4))
    mlflow.log_metric("r2",   round(r2,   4))
    return mae, rmse, r2


print(" Logging Random Forest to MLflow...")

with mlflow.start_run(run_name="random-forest-regressor"):
    mlflow.log_params({
        "n_estimators":      rf_model.n_estimators,
        "max_depth":         str(rf_model.max_depth),
        "min_samples_leaf":  rf_model.min_samples_leaf,
        "min_samples_split": rf_model.min_samples_split,
        "random_state":      42
    })

    rf_preds      = rf_model.predict(X_scaled)
    mae, rmse, r2 = log_regression_metrics(y_true, rf_preds)

   
    mlflow.set_tag("model_type",       "RandomForestRegressor")
    mlflow.set_tag("dataset_version",  "NYC_taxi_jan2024")
    mlflow.set_tag("assignment",       "Assignment4")
    mlflow.set_tag("task",             "tip_amount_regression")

   
    mlflow.sklearn.log_model(
        rf_model,
        "model",
        registered_model_name="taxi-tip-regressor"
    )

    rf_run_id = mlflow.active_run().info.run_id
    print(f" Random Forest logged!")
    print(f"   Run ID: {rf_run_id}")
    print(f"   MAE:  {mae:.4f}")
    print(f"   RMSE: {rmse:.4f}")
    print(f"   R²:   {r2:.4f}")


print("\n Training and logging Linear Regression...")

lr_model = LinearRegression()
lr_model.fit(X_scaled, y_true)

with mlflow.start_run(run_name="linear-regression"):
    mlflow.log_params({
        "model_type":  "LinearRegression",
        "fit_intercept": True,
        "random_state":  42
    })

    lr_preds      = lr_model.predict(X_scaled)
    mae, rmse, r2 = log_regression_metrics(y_true, lr_preds)

   
    mlflow.set_tag("model_type",      "LinearRegression")
    mlflow.set_tag("dataset_version", "NYC_taxi_jan2024")
    mlflow.set_tag("assignment",      "Assignment4")
    mlflow.set_tag("task",            "tip_amount_regression")

    mlflow.sklearn.log_model(lr_model, "model")

    lr_run_id = mlflow.active_run().info.run_id
    print(f" Linear Regression logged!")
    print(f"   Run ID: {lr_run_id}")
    print(f"   MAE:  {mae:.4f}")
    print(f"   RMSE: {rmse:.4f}")
    print(f"   R²:   {r2:.4f}")

print("\n Both models logged to MLflow!")


 Logging Random Forest to MLflow...


2026/04/17 19:01:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/17 19:01:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'taxi-tip-regressor'.
2026/04/17 19:01:56 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: taxi-tip-regressor, version 1
Created version '1' of model 'taxi-tip-regressor'.


 Random Forest logged!
   Run ID: 7c83ee3fb07f4b1ca23cf4cc8a430997
   MAE:  1.9749
   RMSE: 2.6262
   R²:   0.3112
🏃 View run random-forest-regressor at: http://localhost:5000/#/experiments/0/runs/7c83ee3fb07f4b1ca23cf4cc8a430997
🧪 View experiment at: http://localhost:5000/#/experiments/0

 Training and logging Linear Regression...


2026/04/17 19:01:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/17 19:01:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


 Linear Regression logged!
   Run ID: b65870d58b6544d68e6e984555c2cf52
   MAE:  1.4159
   RMSE: 1.7954
   R²:   0.6781
🏃 View run linear-regression at: http://localhost:5000/#/experiments/0/runs/b65870d58b6544d68e6e984555c2cf52
🧪 View experiment at: http://localhost:5000/#/experiments/0

 Both models logged to MLflow!


In [6]:
# ============================================================
# RE-LOG RANDOM FOREST (in case it didn't save properly)
# ============================================================

print(" Logging Random Forest to MLflow...")

with mlflow.start_run(run_name="random-forest-regressor"):
    
    mlflow.log_params({
        "n_estimators":      rf_model.n_estimators,
        "max_depth":         str(rf_model.max_depth),
        "min_samples_leaf":  rf_model.min_samples_leaf,
        "min_samples_split": rf_model.min_samples_split,
        "random_state":      42
    })

    
    rf_preds = rf_model.predict(X_scaled)
    
    mae  = mean_absolute_error(y_true, rf_preds)
    rmse = np.sqrt(mean_squared_error(y_true, rf_preds))
    r2   = r2_score(y_true, rf_preds)
    
   
    mlflow.log_metric("mae",  round(mae,  4))
    mlflow.log_metric("rmse", round(rmse, 4))
    mlflow.log_metric("r2",   round(r2,   4))

    
    mlflow.set_tag("model_type",      "RandomForestRegressor")
    mlflow.set_tag("dataset_version", "NYC_taxi_jan2024")
    mlflow.set_tag("assignment",      "Assignment4")
    mlflow.set_tag("task",            "tip_amount_regression")

    
    mlflow.sklearn.log_model(
        rf_model,
        "model",
        registered_model_name="taxi-tip-regressor"
    )

    rf_run_id = mlflow.active_run().info.run_id

print(f" Random Forest logged successfully!")
print(f"   Run ID: {rf_run_id}")
print(f"   MAE:    ${mae:.4f}")
print(f"   RMSE:   ${rmse:.4f}")
print(f"   R²:     {r2:.4f}")
print(f"\n Go to http://localhost:5000 to see it!")



 Logging Random Forest to MLflow...


2026/04/17 19:07:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/17 19:07:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'taxi-tip-regressor' already exists. Creating a new version of this model...
2026/04/17 19:07:20 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: taxi-tip-regressor, version 2
Created version '2' of model 'taxi-tip-regressor'.


🏃 View run random-forest-regressor at: http://localhost:5000/#/experiments/0/runs/82c09e02ccf142f9b8b8223b6d8990b4
🧪 View experiment at: http://localhost:5000/#/experiments/0
 Random Forest logged successfully!
   Run ID: 82c09e02ccf142f9b8b8223b6d8990b4
   MAE:    $1.9749
   RMSE:   $2.6262
   R²:     0.3112

 Go to http://localhost:5000 to see it!


In [7]:
# ============================================================
# TASK 1.2: MODEL COMPARISON & REGISTRY
# ============================================================

from mlflow.tracking import MlflowClient

client = MlflowClient()

experiment = mlflow.get_experiment_by_name("Default")
if experiment is None:
    experiment = client.get_experiment_by_name("taxi-tip-prediction")

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.mae ASC"]
)

print(" ALL LOGGED RUNS - COMPARISON:")
print("=" * 60)
cols = ['run_id', 'metrics.mae', 'metrics.rmse', 
        'metrics.r2', 'tags.model_type']
available = [c for c in cols if c in runs.columns]
print(runs[available].to_string(index=False))

#Finding best run
best_run = runs.loc[runs['metrics.r2'].idxmax()]
best_run_id = best_run['run_id']

print(f"\n Best model: {best_run.get('tags.model_type', 'Unknown')}")
print(f"   Run ID: {best_run_id}")
print(f"   MAE:  {best_run['metrics.mae']:.4f}")
print(f"   RMSE: {best_run['metrics.rmse']:.4f}")
print(f"   R²:   {best_run['metrics.r2']:.4f}")

 ALL LOGGED RUNS - COMPARISON:
                          run_id  metrics.mae  metrics.rmse  metrics.r2       tags.model_type
b65870d58b6544d68e6e984555c2cf52       1.4159        1.7954      0.6781      LinearRegression
82c09e02ccf142f9b8b8223b6d8990b4       1.9749        2.6262      0.3112 RandomForestRegressor
7c83ee3fb07f4b1ca23cf4cc8a430997       1.9749        2.6262      0.3112 RandomForestRegressor

 Best model: LinearRegression
   Run ID: b65870d58b6544d68e6e984555c2cf52
   MAE:  1.4159
   RMSE: 1.7954
   R²:   0.6781


In [8]:
# ============================================================
#REGISTERING BEST MODEL & LOAD FROM REGISTRY
# ============================================================

from mlflow.tracking import MlflowClient
client = MlflowClient()


best_run_id = "b65870d58b6544d68e6e984555c2cf52"

print(" Registering best model in MLflow Model Registry...")

result = mlflow.register_model(
    model_uri=f"runs:/{best_run_id}/model",
    name="taxi-tip-regressor-best"
)

print(f" Model registered!")
print(f"   Name: {result.name}")
print(f"   Version: {result.version}")


client.update_model_version(
    name="taxi-tip-regressor-best",
    version=result.version,
    description=(
        "Linear Regression model for NYC taxi tip prediction. "
        "Trained on January 2024 NYC Yellow Taxi data (credit card only). "
        f"Performance: MAE=1.4159, RMSE=1.7954, R²=0.6781. "
        "Features: 29 engineered features including fare, distance, time, borough."
    )
)

print(f"\n Version description added!")

client.transition_model_version_stage(
    name="taxi-tip-regressor-best",
    version=result.version,
    stage="Production"
)

print(f" Model promoted to Production stage!")

print(f"\nLoading model back from registry...")

loaded_model = mlflow.sklearn.load_model(
    f"models:/taxi-tip-regressor-best/Production"
)


sample_trip = X_scaled.iloc[[0]]
prediction  = loaded_model.predict(sample_trip)

print(f" Model loaded from registry successfully!")
print(f"\n Sample Prediction:")
print(f"   Input features: {len(sample_trip.columns)} features")
print(f"   Predicted tip amount: ${prediction[0]:.2f}")
print(f"\n Part 1 Complete!")

 Registering best model in MLflow Model Registry...


Successfully registered model 'taxi-tip-regressor-best'.
2026/04/17 19:25:27 WARNING mlflow.tracking._model_registry.fluent: Run with id b65870d58b6544d68e6e984555c2cf52 has no artifacts at artifact path 'model', registering model based on models:/m-a9f8c11f034d47cdaadf4e215c31bf31 instead
2026/04/17 19:25:27 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: taxi-tip-regressor-best, version 1
Created version '1' of model 'taxi-tip-regressor-best'.


 Model registered!
   Name: taxi-tip-regressor-best
   Version: 1

 Version description added!
 Model promoted to Production stage!

Loading model back from registry...
 Model loaded from registry successfully!

 Sample Prediction:
   Input features: 29 features
   Predicted tip amount: $6.14

 Part 1 Complete!


## Part 1 Summary: MLflow Experiment Tracking

### Task 1.1: Experiment Logging
- Created MLflow experiment: "taxi-tip-prediction"  
- Logged 2 models with parameters, metrics and artifacts:
  - Random Forest Regressor: MAE=1.97, RMSE=2.63, R²=0.31
  - Linear Regression: MAE=1.42, RMSE=1.80, R²=0.68
- Tags applied: model_type, dataset_version, assignment, task

### Task 1.2: Model Registry
- Compared all runs side-by-side in MLflow UI (see screenshot)
- **Best model: Linear Regression** (highest R²=0.68 on evaluation sample)
- Registered as "taxi-tip-regressor-best" in MLflow Model Registry
- Promoted to Production stage
- Successfully loaded from registry and made sample prediction

### Screenshots
- mlflow_runs.png: All logged runs in MLflow UI
- mlflow_compare.png: Side-by-side comparison view